# Phase 1: ROI Preprocessing & Fine-tuning
## Objective
1. Preprocess mammogram images with ROI (Region of Interest) extraction
2. Fine-tune the best model (Custom CNN, 61.1% baseline) on ROI-cropped images
3. Optimize threshold for high sensitivity (≥95%)
4. Evaluate all metrics: accuracy, sensitivity, specificity, F1, ROC-AUC

## Step 1: Setup & Imports

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project to path
project_root = Path('/Users/GiangNguyenHuy/Documents/breast-cancer-ai')
sys.path.insert(0, str(project_root))

from src.data_processing.roi_preprocessing import (
    batch_preprocess_images,
    preprocess_mammogram_roi
)
from src.evaluation import (
    calculate_clinical_metrics,
    find_optimal_threshold
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

TensorFlow version: 2.21.0
GPU available: []


## Step 2: Preprocess Images with ROI Crop
This step removes background (black areas) and text labels, keeping only the breast region.

In [3]:
import logging
logging.basicConfig(level=logging.INFO)

base_dir = project_root

# Paths for original and ROI-cropped images
original_dir = base_dir / 'data/cbis_ddsm/processed/images'
roi_dir = base_dir / 'data/cbis_ddsm/processed/images_roi'

print("Preprocessing images with ROI crop...")
print("This may take 5-10 minutes depending on image count.\n")

# Process train set
print("[1/3] Processing training set...")
batch_preprocess_images(
    image_dir=original_dir / 'train',
    output_dir=roi_dir / 'train',
    target_size=(224, 224),
    threshold_value=10,
    margin=20
)

# Process validation set
print("\n[2/3] Processing validation set...")
batch_preprocess_images(
    image_dir=original_dir / 'val',
    output_dir=roi_dir / 'val',
    target_size=(224, 224),
    threshold_value=10,
    margin=20
)

# Process test set
print("\n[3/3] Processing test set...")
batch_preprocess_images(
    image_dir=original_dir / 'test',
    output_dir=roi_dir / 'test',
    target_size=(224, 224),
    threshold_value=10,
    margin=20
)

print("\n✅ ROI preprocessing complete!")

Preprocessing images with ROI crop...
This may take 5-10 minutes depending on image count.

[1/3] Processing training set...


INFO:src.data_processing.roi_preprocessing:Processed 100 images...
INFO:src.data_processing.roi_preprocessing:Processed 200 images...
INFO:src.data_processing.roi_preprocessing:Processed 300 images...
INFO:src.data_processing.roi_preprocessing:Processed 400 images...
INFO:src.data_processing.roi_preprocessing:Processed 500 images...
INFO:src.data_processing.roi_preprocessing:Processed 600 images...
INFO:src.data_processing.roi_preprocessing:Processed 700 images...
INFO:src.data_processing.roi_preprocessing:Processed 800 images...
INFO:src.data_processing.roi_preprocessing:Processed 900 images...
INFO:src.data_processing.roi_preprocessing:Processed 1000 images...
INFO:src.data_processing.roi_preprocessing:Processed 1100 images...
INFO:src.data_processing.roi_preprocessing:Processed 1200 images...
INFO:src.data_processing.roi_preprocessing:Processed 1300 images...
INFO:src.data_processing.roi_preprocessing:Processed 1400 images...
INFO:src.data_processing.roi_preprocessing:Processed 1500


[2/3] Processing validation set...


INFO:src.data_processing.roi_preprocessing:Processed 100 images...
INFO:src.data_processing.roi_preprocessing:Processed 200 images...
INFO:src.data_processing.roi_preprocessing:Processed 300 images...
INFO:src.data_processing.roi_preprocessing:Preprocessing complete: 383 successful, 0 errors



[3/3] Processing test set...


INFO:src.data_processing.roi_preprocessing:Processed 100 images...
INFO:src.data_processing.roi_preprocessing:Processed 200 images...
INFO:src.data_processing.roi_preprocessing:Processed 300 images...
INFO:src.data_processing.roi_preprocessing:Preprocessing complete: 386 successful, 0 errors



✅ ROI preprocessing complete!


## Step 3: Load ROI-Cropped Data

In [4]:
def load_images_from_directory(directory, label, max_images=None):
    """Load images from directory with given label."""
    images = []
    labels = []
    
    dir_path = Path(directory)
    image_files = sorted(dir_path.glob('*.png'))
    
    if max_images:
        image_files = image_files[:max_images]
    
    for img_file in image_files:
        try:
            img = keras.preprocessing.image.load_img(
                str(img_file),
                target_size=(224, 224)
            )
            img_array = keras.preprocessing.image.img_to_array(img)
            img_array = img_array / 255.0  # Normalize to 0-1
            images.append(img_array)
            labels.append(label)
        except Exception as e:
            print(f'Error loading {img_file}: {e}')
            continue
    
    return np.array(images), np.array(labels)

# Load training data
print("Loading training data...")
train_benign, _ = load_images_from_directory(
    roi_dir / 'train' / 'benign', 0
)
train_malignant, _ = load_images_from_directory(
    roi_dir / 'train' / 'malignant', 1
)
X_train = np.concatenate([train_benign, train_malignant])
y_train = np.concatenate([np.zeros(len(train_benign)), np.ones(len(train_malignant))])

# Load validation data
print("Loading validation data...")
val_benign, _ = load_images_from_directory(
    roi_dir / 'val' / 'benign', 0
)
val_malignant, _ = load_images_from_directory(
    roi_dir / 'val' / 'malignant', 1
)
X_val = np.concatenate([val_benign, val_malignant])
y_val = np.concatenate([np.zeros(len(val_benign)), np.ones(len(val_malignant))])

# Shuffle
from sklearn.utils import shuffle
X_train, y_train = shuffle(X_train, y_train, random_state=42)
X_val, y_val = shuffle(X_val, y_val, random_state=42)

print(f"\nTrain set: {X_train.shape}, {np.sum(y_train==1)}/{np.sum(y_train==0)} malignant/benign")
print(f"Val set: {X_val.shape}, {np.sum(y_val==1)}/{np.sum(y_val==0)} malignant/benign")

Loading training data...
Loading validation data...

Train set: (1790, 224, 224, 3), 750/1040 malignant/benign
Val set: (383, 224, 224, 3), 160/223 malignant/benign


## Step 4: Load Pre-trained Custom CNN and Fine-tune

In [6]:
# Load the best model (Custom CNN)
model_path = project_root / 'models/deep_learning/custom_cnn_best.keras'

print(f"Loading model from {model_path}...")
model = keras.models.load_model(str(model_path))
model.summary()

Loading model from /Users/GiangNguyenHuy/Documents/breast-cancer-ai/models/deep_learning/custom_cnn_best.keras...


Model: "Custom_CNN_Mammogram"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 4,320,101 (16.48 MB)

 Trainable params: 1,438,881 (5.49 MB)

 Non-trainable params: 3,456 (13.50 KB)

 Optimizer params: 2,877,764 (10.98 MB)

### Fine-tune with ROI-cropped data
We'll unfreeze the last few layers and train with a low learning rate.

In [7]:
# Unfreeze last 10 layers for fine-tuning
for layer in model.layers[:-10]:
    layer.trainable = False

for layer in model.layers[-10:]:
    layer.trainable = True

# Use low learning rate for fine-tuning
optimizer = keras.optimizers.Adam(learning_rate=1e-5)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights])}")
print("Ready to fine-tune!")

Trainable parameters: 264705
Ready to fine-tune!


In [8]:
# Fine-tune for 10 epochs
print("Fine-tuning Custom CNN on ROI-cropped images...")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

print("\n✅ Fine-tuning complete!")

Fine-tuning Custom CNN on ROI-cropped images...
Epoch 1/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.5313 - loss: 0.7634 - val_accuracy: 0.5170 - val_loss: 0.7172
Epoch 2/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.5335 - loss: 0.7599 - val_accuracy: 0.5274 - val_loss: 0.7089
Epoch 3/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.5341 - loss: 0.7533 - val_accuracy: 0.5248 - val_loss: 0.7048
Epoch 4/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.5285 - loss: 0.7400 - val_accuracy: 0.5196 - val_loss: 0.7034
Epoch 5/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 67s 1s/step - accuracy: 0.5307 - loss: 0.7369 - val_accuracy: 0.5222 - val_loss: 0.7013
Epoch 6/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.5134 - loss: 0.7244 - val_accuracy: 0.5065 - val_loss: 0.6996
Epoch 7/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.5307 - loss: 0.7188 - val_accuracy: 0.5144 - val_loss: 0.6983
Epoch 8/10
56/56 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.5246 -

### Save fine-tuned model

In [9]:
# Save fine-tuned model
finetuned_model_path = project_root / 'models/deep_learning/custom_cnn_v2_finetuned_roi.keras'
model.save(str(finetuned_model_path))
print(f"✅ Model saved to {finetuned_model_path}")

# Also save training history
import json
history_dict = {k: [float(v) for v in val] for k, val in history.history.items()}
with open(project_root / 'experiments/results/custom_cnn_finetuning_history.json', 'w') as f:
    json.dump(history_dict, f, indent=2)
print("✅ Training history saved")

✅ Model saved to /Users/GiangNguyenHuy/Documents/breast-cancer-ai/models/deep_learning/custom_cnn_v2_finetuned_roi.keras
✅ Training history saved


## Step 5: Evaluate Model & Optimize Threshold for High Sensitivity

In [10]:
# Get predictions on validation set
print("Generating predictions on validation set...")
y_pred_proba = model.predict(X_val, verbose=0).flatten()

print(f"Probability range: [{y_pred_proba.min():.4f}, {y_pred_proba.max():.4f}]")
print(f"Mean probability: {y_pred_proba.mean():.4f}")
print(f"Median probability: {np.median(y_pred_proba):.4f}")

Generating predictions on validation set...
Probability range: [0.3090, 0.7166]
Mean probability: 0.5213
Median probability: 0.5563


In [11]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, precision_recall_curve
)

# Find threshold that maximizes sensitivity (recall/TPR) ≥ 95%
# while also considering a reasonable specificity

thresholds = np.arange(0.0, 1.01, 0.01)
results = []

for threshold in thresholds:
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    accuracy = accuracy_score(y_val, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = f1_score(y_val, y_pred, zero_division=0)
    
    results.append({
        'threshold': threshold,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'accuracy': accuracy,
        'precision': precision,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 threshold  sensitivity  specificity  accuracy  precision       f1  tp  fp  tn  fn
      0.00      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.01      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.02      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.03      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.04      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.05      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.06      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.07      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.08      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.09      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
      0.10      1.00000     0.000000  0.417755   0.417755 0.589319 160 223   0   0
    

In [12]:
# Find best threshold for sensitivity ≥ 95%
high_sensitivity = df_results[df_results['sensitivity'] >= 0.95]

if len(high_sensitivity) > 0:
    # Among high sensitivity options, choose the one with best F1
    best_idx = high_sensitivity['f1'].idxmax()
    best_threshold = df_results.loc[best_idx]
    print("\n=== OPTIMAL THRESHOLD FOR SENSITIVITY ≥ 95% ===")
    print(f"Threshold: {best_threshold['threshold']:.2f}")
    print(f"Sensitivity: {best_threshold['sensitivity']:.4f}")
    print(f"Specificity: {best_threshold['specificity']:.4f}")
    print(f"Accuracy: {best_threshold['accuracy']:.4f}")
    print(f"Precision: {best_threshold['precision']:.4f}")
    print(f"F1: {best_threshold['f1']:.4f}")
else:
    print("\n⚠️ Cannot achieve sensitivity ≥ 95%")
    print("Choosing threshold with maximum sensitivity instead...")
    best_idx = df_results['sensitivity'].idxmax()
    best_threshold = df_results.loc[best_idx]
    print(f"\nThreshold: {best_threshold['threshold']:.2f}")
    print(f"Sensitivity: {best_threshold['sensitivity']:.4f}")


=== OPTIMAL THRESHOLD FOR SENSITIVITY ≥ 95% ===
Threshold: 0.39
Sensitivity: 0.9625
Specificity: 0.1256
Accuracy: 0.4752
Precision: 0.4413
F1: 0.6051


## Step 6: Final Evaluation on Test Set

In [13]:
# Load test set
print("Loading test set...")
test_benign, _ = load_images_from_directory(
    roi_dir / 'test' / 'benign', 0
)
test_malignant, _ = load_images_from_directory(
    roi_dir / 'test' / 'malignant', 1
)
X_test = np.concatenate([test_benign, test_malignant])
y_test = np.concatenate([np.zeros(len(test_benign)), np.ones(len(test_malignant))])

# Shuffle
X_test, y_test = shuffle(X_test, y_test, random_state=42)

print(f"Test set: {X_test.shape}, {np.sum(y_test==1)}/{np.sum(y_test==0)} malignant/benign")

Loading test set...
Test set: (386, 224, 224, 3), 162/224 malignant/benign


In [14]:
# Get test predictions
print("Generating predictions on test set...")
y_test_proba = model.predict(X_test, verbose=0).flatten()

# Apply optimal threshold
optimal_threshold = best_threshold['threshold']
y_test_pred = (y_test_proba >= optimal_threshold).astype(int)

# Calculate metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
accuracy = accuracy_score(y_test, y_test_pred)
precision = tp / (tp + fp)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_proba)

print("\n=== TEST SET RESULTS ===")
print(f"Threshold: {optimal_threshold:.2f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Sensitivity (TPR): {sensitivity:.4f}")
print(f"Specificity (TNR): {specificity:.4f}")
print(f"Precision (PPV): {precision:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP: {tp}, FP: {fp}")
print(f"  FN: {fn}, TN: {tn}")

Generating predictions on test set...

=== TEST SET RESULTS ===
Threshold: 0.39
Accuracy: 0.4793
Sensitivity (TPR): 0.9506
Specificity (TNR): 0.1384
Precision (PPV): 0.4438
F1 Score: 0.6051
ROC-AUC: 0.5836

Confusion Matrix:
  TP: 154, FP: 193
  FN: 8, TN: 31


## Step 7: Save Results

In [15]:
import json
from datetime import datetime

# Prepare results summary
results_summary = {
    'timestamp': datetime.now().isoformat(),
    'model': 'Custom CNN v2 (Fine-tuned on ROI-cropped images)',
    'preprocessing': 'ROI extraction + thresholding + margin=20px',
    'finetuning': {
        'epochs': 10,
        'learning_rate': 1e-5,
        'unfrozen_layers': 10
    },
    'threshold_optimization': {
        'method': 'Grid search (0.0-1.0, step=0.01)',
        'target': 'Sensitivity ≥ 95%'
    },
    'validation_metrics': {
        'threshold': float(best_threshold['threshold']),
        'accuracy': float(best_threshold['accuracy']),
        'sensitivity': float(best_threshold['sensitivity']),
        'specificity': float(best_threshold['specificity']),
        'precision': float(best_threshold['precision']),
        'f1': float(best_threshold['f1'])
    },
    'test_metrics': {
        'accuracy': float(accuracy),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'precision': float(precision),
        'f1': float(f1),
        'roc_auc': float(roc_auc),
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'tn': int(tn)
    }
}

# Save results
results_file = project_root / 'experiments/results/phase1_results.json'
with open(results_file, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"✅ Results saved to {results_file}")

# Save threshold analysis
csv_file = project_root / 'experiments/results/phase1_threshold_analysis.csv'
df_results.to_csv(csv_file, index=False)
print(f"✅ Threshold analysis saved to {csv_file}")

✅ Results saved to /Users/GiangNguyenHuy/Documents/breast-cancer-ai/experiments/results/phase1_results.json
✅ Threshold analysis saved to /Users/GiangNguyenHuy/Documents/breast-cancer-ai/experiments/results/phase1_threshold_analysis.csv
